In [ ]:
print("Installing libraries...")
!pip install -q sentence-transformers langchain-chroma langchain-community google-genai pypdf
print("Done!")

Installing libraries...
Done!


In [ ]:
# Import all the tools we need
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google import genai
from google.colab import drive
import os
import shutil

print("✅ Libraries imported!")

# Connect to Google Drive
print("Connecting to Google Drive...")
drive.mount('/content/drive')

print("✅ All set!")

✅ Libraries imported!
Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All set!


In [ ]:
from google.colab import userdata  # keys are stored in Colab Secrets
# Set up Gemini AI
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# Tell the system where your PDFs are
pdf_folder = "/content/drive/MyDrive/Ramadan_RAG_Data/"


In [ ]:
# Where to save the database
db_path = "/content/drive/MyDrive/Ramadan_RAG_Data/ramadan_db_v0"

print(" Configuration set!")
print(f" Looking for PDFs in: {pdf_folder}")

 Configuration set!
 Looking for PDFs in: /content/drive/MyDrive/Ramadan_RAG_Data/


In [ ]:
# Check if folder exists
if not os.path.exists(pdf_folder):
    print(" ERROR: Folder not found!")
    print(f"Please check the path: {pdf_folder}")
else:
    print(f" Folder found!")

    # List all PDF files
    print("\n Scanning for PDFs...")
    all_documents = []
    pdf_count = 0

    for filename in os.listdir(pdf_folder):
        if filename.endswith(".pdf"):# ingnore anything that isn't a PDF
            pdf_count += 1
            file_path = os.path.join(pdf_folder, filename)
            print(f"    Loading: {filename}")

            try:
                loader = PyPDFLoader(file_path)
                all_documents.extend(loader.load())
            except Exception as e:
                print(f"    Error reading {filename}: {e}")

    if pdf_count == 0:
        print("\n No PDF files found in folder!")
    else:
        print(f"\n Successfully loaded {len(all_documents)} pages from {pdf_count} PDF(s)")

 Folder found!

 Scanning for PDFs...
    Loading: IDF_DaR_Practical_Guidelines_Ramadan.pdf
    Loading: s-0045-1814095.pdf
    Loading: s-0045-1813010.pdf
    Loading: circulaire RAMADAN 2026.pdf
    Loading: score ramadan.pdf
    Loading: Category.pdf

 Successfully loaded 362 pages from 6 PDF(s)


In [ ]:
# Split the text into smaller chunks
print(" Breaking text into searchable chunks...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Each chunk is ~1000 characters
    chunk_overlap=200     # Overlap to keep context
)

chunks = text_splitter.split_documents(all_documents)

print(f" Created {len(chunks)} chunks")

 Breaking text into searchable chunks...
 Created 1049 chunks


In [ ]:
# Create a class that converts text to numbers
class SentenceTransformerEmbeddings:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        # Convert multiple texts to numbers
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        # Convert a single question to numbers
        return self.model.encode([text])[0].tolist()

print(" Embedding class ready!")

 Embedding class ready!


In [ ]:
# Check if database already exists
if os.path.exists(db_path):
    print(" Database already exists!")
    print("Deleting old version to build fresh...")
    shutil.rmtree(db_path)

# Create the embedding model
print(" Building the vector database...")
print("(This takes 1-2 minutes - please wait)")

embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Build and save the database
db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_path
)

print("\n" + "=" * 60)
print(f" DATABASE COMPLETE!")
print(f" Stored {len(chunks)} chunks")
print(f" Saved to: {db_path}")
print("=" * 60)

 Building the vector database...
(This takes 1-2 minutes - please wait)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 DATABASE COMPLETE!
 Stored 1049 chunks
 Saved to: /content/drive/MyDrive/Ramadan_RAG_Data/ramadan_db_v0
